# Semana 12 · Integración de datos
## NovaMarket: ventas + bodega + CRM en un solo dataset

**Ricardo Borja** · Preprocesamiento de Datos · Misión 3

Esta semana no se limpia ni se transforma nada nuevo. Se integran tres fuentes que viven en
sistemas distintos y que nombran lo mismo de maneras distintas:

| Fuente | Archivo | Grano (qué es una fila) |
|---|---|---|
| Ventas | `S11_PD_Grupo05_DatasetTransformado.csv` | un pedido |
| Bodega | `bodega_categorias_novamarket.csv` | una categoría de producto |
| CRM | `crm_clientes_novamarket.csv` | un cliente inscrito al club |

La integración es horizontal: se agregan columnas, no filas. Después de cada join se compara el
número de filas antes y después, porque un join mal hecho no lanza error, solo pierde o duplica
pedidos en silencio.

In [1]:
import unicodedata
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

RUTA_VENTAS = "S11_PD_Grupo05_DatasetTransformado.csv"
RUTA_BODEGA = "bodega_categorias_novamarket.csv"
RUTA_CRM    = "crm_clientes_novamarket.csv"
RUTA_SALIDA = "S12_PD_Borja_DatasetIntegrado.csv"

## 1. Cargar las tres fuentes y revisar su forma

In [2]:
df_ventas = pd.read_csv(RUTA_VENTAS, low_memory=False)
df_bodega = pd.read_csv(RUTA_BODEGA)
df_crm    = pd.read_csv(RUTA_CRM)

pd.DataFrame(
    [{"fuente": "ventas", "filas": df_ventas.shape[0], "columnas": df_ventas.shape[1]},
     {"fuente": "bodega", "filas": df_bodega.shape[0], "columnas": df_bodega.shape[1]},
     {"fuente": "CRM",    "filas": df_crm.shape[0],    "columnas": df_crm.shape[1]}]
)

,fuente,filas,columnas
0,ventas,620000,116
1,bodega,7,5
2,CRM,411840,6


In [3]:
df_bodega

,categoria,proveedor_principal,bodega_asignada,dias_reposicion,stock_minimo
0,Electrónica,TecnoAndina S.A.S.,Bodega Norte,7,120
1,Hogar,Distribuidora El Roble,Bodega Centro,14,200
2,Moda,Textiles Caribe Ltda.,Bodega Norte,21,350
3,Deportes,Deportes del Valle S.A.,Bodega Sur,10,150
4,Juguetería,Juguetes Arcoíris,Bodega Centro,30,90
5,Papelería,Papeles y Más S.A.S.,Bodega Sur,12,400
6,Mascotas,AnimalCare Colombia,Bodega Norte,9,110


In [4]:
df_crm.head()

,email_cliente,ciudad_residencia,fecha_inscripcion_club,puntos_fidelidad_actuales,segmento_crm,acepta_promociones
0,cliente263677@outlook.com,Cartagena,2024-05-05,9306,Inactivo,No
1,cliente536992@gmail.com,Santa Marta,2025-12-17,1067,Inactivo,Si
2,cliente204522@yahoo.es,Cali,2023-12-25,4137,Recurrente,No
3,cliente264909@novamarket.com.co,Cúcuta,2026-06-22,38,Recurrente,Si
4,cliente16065@gmail.com,Cúcuta,2023-08-24,9397,Inactivo,No


## 2. Un obstáculo previo: la llave de categoría ya no es texto

El dataset de ventas pasó por la codificación de la Semana 9, así que `categoria_producto` y
`ciudad_tienda` ya no existen como columnas de texto: quedaron convertidas en variables indicadoras,
una columna por valor. No se puede hacer un join sobre una columna que ya no está.

Antes de integrar hay que reconstruirlas. La reconstrucción es exacta porque la codificación fue
one-hot completa: cada fila tiene exactamente un 1 entre las seis columnas de categoría y
exactamente un 1 entre las doce de ciudad, así que no se perdió ninguna información al codificar.

In [5]:
cols_categoria = [c for c in df_ventas.columns if c.startswith("categoria_producto_")]
cols_ciudad = [c for c in df_ventas.columns
               if c.startswith("ciudad_tienda_") and c != "ciudad_tienda_imputada"]

pd.DataFrame(
    [{"grupo": "categoria_producto", "columnas_indicadoras": len(cols_categoria),
      "filas_con_exactamente_un_1": int((df_ventas[cols_categoria].sum(axis=1) == 1).sum())},
     {"grupo": "ciudad_tienda", "columnas_indicadoras": len(cols_ciudad),
      "filas_con_exactamente_un_1": int((df_ventas[cols_ciudad].sum(axis=1) == 1).sum())}]
)

,grupo,columnas_indicadoras,filas_con_exactamente_un_1
0,categoria_producto,6,620000
1,ciudad_tienda,12,620000


In [6]:
def reconstruir_desde_indicadoras(df, columnas, prefijo):
    """Devuelve la etiqueta original a partir de un bloque de columnas one-hot."""
    etiquetas = [c[len(prefijo):] for c in columnas]
    posicion = df[columnas].to_numpy().argmax(axis=1)
    return pd.Series([etiquetas[i] for i in posicion], index=df.index)


df_ventas["categoria_producto"] = reconstruir_desde_indicadoras(
    df_ventas, cols_categoria, "categoria_producto_")
df_ventas["ciudad_tienda"] = reconstruir_desde_indicadoras(
    df_ventas, cols_ciudad, "ciudad_tienda_")

df_ventas[["id_pedido", "correo_cliente", "categoria_producto", "ciudad_tienda"]].head()

,id_pedido,correo_cliente,categoria_producto,ciudad_tienda
0,P505195,cliente405195@novamarket.com.co,moda,bogota
1,P677836,cliente577836@yahoo.es,electronica,manizales
2,P387869,cliente287869@novamarket.com.co,jugueteria,cucuta
3,P481069,cliente381069@gmail.com,deportes,cucuta
4,P247920,cliente147920@novamarket.com.co,hogar,bucaramanga


## 3. Conflictos de esquema entre las tres fuentes

Los nombres de columna no coinciden entre sistemas. Ninguno está mal: cada equipo diseñó el suyo
por su cuenta.

| Concepto del mundo real | En ventas | En la otra fuente | Tipo de conflicto |
|---|---|---|---|
| Categoría del producto | `categoria_producto` | `categoria` (bodega) | De nombre |
| Correo del cliente | `correo_cliente` | `email_cliente` (CRM) | De nombre |
| Ciudad | `ciudad_tienda` | `ciudad_residencia` (CRM) | **De valor** |

Los dos primeros se resuelven renombrando antes del join. El tercero no se resuelve renombrando:
`ciudad_tienda` es dónde ocurrió la compra y `ciudad_residencia` es dónde vive el cliente según su
inscripción al club. Un cliente de Cali pudo comprar en la tienda de Bogotá durante un viaje y
ambos valores serían correctos. Las dos columnas se conservan por separado.

Hay además un conflicto estructural: el grano de cada tabla es distinto (pedido, categoría,
cliente). Al unir con `how="left"` desde ventas, el resultado conserva el grano de pedido y la
información de bodega y de CRM se repite entre pedidos del mismo cliente o de la misma categoría.
Eso es correcto y esperado.

## 4. Verificar que las llaves no estén repetidas antes de unir

In [7]:
pd.DataFrame(
    [{"tabla": "bodega", "llave": "categoria",
      "valores_unicos": df_bodega["categoria"].nunique(),
      "duplicados": int(df_bodega["categoria"].duplicated().sum())},
     {"tabla": "CRM", "llave": "email_cliente",
      "valores_unicos": df_crm["email_cliente"].nunique(),
      "duplicados": int(df_crm["email_cliente"].duplicated().sum())}]
)

,tabla,llave,valores_unicos,duplicados
0,bodega,categoria,7,0
1,CRM,email_cliente,411840,0


Si alguna de las dos llaves estuviera repetida, el join multiplicaría pedidos: un pedido se
convertiría en tantas filas como veces aparezca la llave en la tabla de la derecha. Con cero
duplicados en ambas, el número de filas tiene que quedar igual después de cada join.

## 5. Estandarizar las llaves: resolución de entidades

Las llaves se escriben distinto entre sistemas aunque se refieran a la misma categoría real. En
ventas la categoría quedó sin tildes durante la codificación (`electronica`, `jugueteria`), y en
bodega viene con tilde (`Electrónica`, `Juguetería`). Comparadas tal cual, esas dos no coinciden y
el join las dejaría sin pareja aunque sean la misma categoría.

Por eso la estandarización quita espacios, pasa a minúscula y además elimina tildes. Sin ese
último paso se perderían dos de las cinco categorías que sí existen en ambas tablas.

In [8]:
def estandarizar_llave(serie):
    """Quita espacios, pasa a minuscula y elimina tildes para poder comparar entre sistemas."""
    texto = serie.astype(str).str.strip().str.lower()
    return (texto.map(lambda s: unicodedata.normalize("NFD", s))
                 .str.encode("ascii", "ignore").str.decode("utf-8"))


df_ventas["categoria_producto"] = estandarizar_llave(df_ventas["categoria_producto"])
df_ventas["correo_cliente"] = estandarizar_llave(df_ventas["correo_cliente"])
df_bodega["categoria"] = estandarizar_llave(df_bodega["categoria"])
df_crm["email_cliente"] = estandarizar_llave(df_crm["email_cliente"])

pd.DataFrame({"categorias_en_ventas": sorted(df_ventas["categoria_producto"].unique())}).join(
    pd.DataFrame({"categorias_en_bodega": sorted(df_bodega["categoria"].unique())}), how="outer"
)

,categorias_en_ventas,categorias_en_bodega
0,belleza,deportes
1,deportes,electronica
2,electronica,hogar
3,hogar,jugueteria
4,jugueteria,mascotas
5,moda,moda
6,NaN,papeleria


## 6. Primer join: ventas con bodega

La llave es la categoría del producto. Se renombra la columna de bodega para que coincida con la
de ventas y se usa `how="left"` para no perder ningún pedido.

In [9]:
df_bodega = df_bodega.rename(columns={"categoria": "categoria_producto"})
filas_antes = df_ventas.shape[0]
df_paso1 = pd.merge(df_ventas, df_bodega, on="categoria_producto", how="left")

pd.DataFrame([{"filas_antes": filas_antes, "filas_despues": df_paso1.shape[0],
               "se_mantuvo": filas_antes == df_paso1.shape[0],
               "columnas_antes": df_ventas.shape[1], "columnas_despues": df_paso1.shape[1]}])

,filas_antes,filas_despues,se_mantuvo,columnas_antes,columnas_despues
0,620000,620000,True,118,122


## 7. Segundo join: el resultado con el CRM

La llave es el correo del cliente. Se renombra `email_cliente` a `correo_cliente` y se usa otra vez
`how="left"`, porque solo están en el CRM los clientes que se inscribieron al club.

In [10]:
df_crm = df_crm.rename(columns={"email_cliente": "correo_cliente"})
filas_antes_crm = df_paso1.shape[0]
df_integrado = pd.merge(df_paso1, df_crm, on="correo_cliente", how="left")

pd.DataFrame([{"filas_antes": filas_antes_crm, "filas_despues": df_integrado.shape[0],
               "se_mantuvo": filas_antes_crm == df_integrado.shape[0],
               "columnas_antes": df_paso1.shape[1], "columnas_despues": df_integrado.shape[1]}])

,filas_antes,filas_despues,se_mantuvo,columnas_antes,columnas_despues
0,620000,620000,True,122,127


## 8. Cuántos pedidos quedaron sin coincidencia, y por qué

In [11]:
col_bodega = "proveedor_principal"
col_crm = [c for c in df_crm.columns if c != "correo_cliente"][0]

sin_bodega = int(df_integrado[col_bodega].isna().sum())
sin_crm = int(df_integrado[col_crm].isna().sum())
total = df_integrado.shape[0]

pd.DataFrame(
    [{"sin_coincidencia_en": "bodega", "pedidos": sin_bodega,
      "porcentaje": round(100 * sin_bodega / total, 2)},
     {"sin_coincidencia_en": "CRM", "pedidos": sin_crm,
      "porcentaje": round(100 * sin_crm / total, 2)}]
)

,sin_coincidencia_en,pedidos,porcentaje
0,bodega,103782,16.74
1,CRM,208160,33.57


In [12]:
cat_ventas = set(df_ventas["categoria_producto"].unique())
cat_bodega = set(df_bodega["categoria_producto"].unique())

pd.DataFrame(
    [{"situacion": "categorias que se venden y si estan en bodega",
      "valores": ", ".join(sorted(cat_ventas & cat_bodega))},
     {"situacion": "se venden pero NO estan en bodega",
      "valores": ", ".join(sorted(cat_ventas - cat_bodega)) or "(ninguna)"},
     {"situacion": "estan en bodega pero NUNCA se han vendido",
      "valores": ", ".join(sorted(cat_bodega - cat_ventas)) or "(ninguna)"}]
)

,situacion,valores
0,categorias que se venden y si estan en bodega,"deportes, electronica, hogar, jugueteria, moda"
1,se venden pero NO estan en bodega,belleza
2,estan en bodega pero NUNCA se han vendido,"mascotas, papeleria"


In [13]:
df_integrado.loc[df_integrado[col_bodega].isna(), "categoria_producto"].value_counts()

categoria_producto
belleza    103782
Name: count, dtype: int64

**Causa de las no coincidencias con bodega.** Los pedidos sin datos de bodega son todos de la misma
categoría, una que Logística todavía no registró en su catálogo. No es un error del join ni del
archivo: es una categoría que el área de ventas sí maneja y el área de bodega aún no. En sentido
contrario, el catálogo tiene categorías que nunca se han vendido, y por eso no aparecen en el
resultado: `how="left"` solo trae las filas de bodega que encuentran pedido.

**Causa de las no coincidencias con CRM.** El CRM solo contiene a los clientes que se inscribieron
al club de fidelización. Quien compró sin inscribirse no tiene fila allí, así que sus columnas de
CRM quedan vacías. Es un vacío esperado y se cuantifica, no se corrige: rellenar esos campos
inventaría clientes que no existen en el club.

## 9. El conflicto de valor: dos ciudades que no son la misma pregunta

`ciudad_tienda` responde dónde se hizo la compra. `ciudad_residencia` responde dónde vive el
cliente. Para un mismo pedido pueden traer valores distintos y las dos ser correctas. Ninguna se
elimina ni se sobrescribe con la otra.

In [14]:
col_ciudad_crm = "ciudad_residencia" if "ciudad_residencia" in df_integrado.columns else None

if col_ciudad_crm:
    con_ambas = df_integrado.dropna(subset=[col_ciudad_crm])
    coinciden = (estandarizar_llave(con_ambas["ciudad_tienda"])
                 == estandarizar_llave(con_ambas[col_ciudad_crm]))
    resumen_ciudades = pd.DataFrame(
        [{"pedidos_con_las_dos_ciudades": len(con_ambas),
          "misma_ciudad": int(coinciden.sum()),
          "ciudad_distinta": int((~coinciden).sum()),
          "porcentaje_distinta": round(100 * (~coinciden).sum() / len(con_ambas), 2)}])
else:
    resumen_ciudades = pd.DataFrame([{"nota": "el CRM no trae ciudad_residencia"}])

resumen_ciudades

,pedidos_con_las_dos_ciudades,misma_ciudad,ciudad_distinta,porcentaje_distinta
0,411840,31416,380424,92.37


In [15]:
df_integrado[["id_pedido", "ciudad_tienda", col_ciudad_crm]].dropna().head(10)

,id_pedido,ciudad_tienda,ciudad_residencia
0,P505195,bogota,Ibagué
1,P677836,manizales,Manizales
2,P387869,cucuta,Bucaramanga
3,P481069,cucuta,Ibagué
5,P465514,medellin,Pereira
6,P684469,pereira,Barranquilla
7,P583062,medellin,Bucaramanga
8,P630170,manizales,Bogotá
9,P622879,cali,Villavicencio
10,P318916,bogota,Medellín


## 10. Revisión de redundancia tras los joins

Se revisa si algún join dejó columnas duplicadas con sufijos `_x` y `_y`, que es lo que ocurre
cuando las dos tablas traen una columna con el mismo nombre que no es la llave.

In [16]:
sufijadas = [c for c in df_integrado.columns if c.endswith("_x") or c.endswith("_y")]

pd.DataFrame([{"columnas_totales": df_integrado.shape[1],
               "columnas_con_sufijo_x_o_y": len(sufijadas),
               "cuales": ", ".join(sufijadas) or "(ninguna)"}])

,columnas_totales,columnas_con_sufijo_x_o_y,cuales
0,127,0,(ninguna)


In [17]:
pd.DataFrame({"columna_nueva_tras_integrar":
              [c for c in df_integrado.columns if c not in df_ventas.columns]})

,columna_nueva_tras_integrar
0,proveedor_principal
1,bodega_asignada
2,dias_reposicion
3,stock_minimo
4,ciudad_residencia
5,fecha_inscripcion_club
6,puntos_fidelidad_actuales
7,segmento_crm
8,acepta_promociones


## 11. Exportar el dataset integrado

In [18]:
df_integrado.to_csv(RUTA_SALIDA, index=False)

pd.DataFrame([{"archivo": RUTA_SALIDA,
               "filas": df_integrado.shape[0],
               "columnas": df_integrado.shape[1],
               "filas_iguales_a_ventas": df_integrado.shape[0] == df_ventas.shape[0]}])

,archivo,filas,columnas,filas_iguales_a_ventas
0,S12_PD_Borja_DatasetIntegrado.csv,620000,127,True


## Cierre

El dataset integrado conserva un pedido por fila, los mismos 620.000 pedidos con que empezó, y suma
las columnas de bodega y de CRM. Los dos joins se hicieron con `how="left"` desde ventas, así que
ningún pedido se perdió, y las llaves se verificaron sin duplicados antes de unir, así que ninguno
se multiplicó.

Lo que queda documentado y sin corregir, a propósito:

- Los pedidos de la categoría que Logística no tiene en catálogo quedan sin datos de bodega.
- Los pedidos de clientes no inscritos al club quedan sin datos de CRM.
- `ciudad_tienda` y `ciudad_residencia` conviven como columnas separadas.

Sobre eso se construyen las características nuevas de la próxima semana.